# 03. FE 関数カタログ (`src/03_fe_all.py`)

各モデルが個別に実装していた FE 関数を 1 ファイルに集約したもの。
「あるモデルでは試したが別のモデルでは未適用」という取りこぼしを探すために作られた。

本番で実際に使われるのはモデル別の `src/03_fe_lgbm.py` / `03_fe_xgb.py` / `03_fe_catboost.py` /
`03_fe_realmlp.py`。`03_fe_all.py` は横断比較用のカタログ。

In [1]:
import os, sys, importlib
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

# `src/03_fe_all.py` のように数字で始まるファイルは `import 03_fe_all` と書けない
# (Python の識別子にならない)。importlib なら文字列で読み込める。
def load_fe(name):
    """name="all" なら src/03_fe_all.py を読み込んで返す。"""
    return importlib.import_module(f"03_fe_{name}")

cwd: C:\Users\takac\dev\python\kaggle\2609


In [2]:
import inspect
import numpy as np
import pandas as pd

fe_all = load_fe("all")
funcs = [(n, f) for n, f in vars(fe_all).items()
         if callable(f) and not n.startswith("_") and getattr(f, "__module__", "") == "03_fe_all"]
pd.DataFrame(
    [(n, str(inspect.signature(f)), (f.__doc__ or "").strip().split("\n")[0]) for n, f in funcs],
    columns=["関数", "シグネチャ", "説明"]
)

,関数,シグネチャ,説明
0,load_data,(data_dir: 'str' = 'data'),
1,get_y,(train: 'pd.DataFrame') -> 'np.ndarray',
2,as_native_category,"(tr, te, cols=None)",train/test 共通のカテゴリ集合で category dtype 化 (LightG...
3,as_ordinal,"(tr, te, cols=None)",整数コード化 (XGBoost の最終採用方式).
4,as_str,"(frames, cols) -> 'None'",文字列化 (CatBoost cat_features / catify 用). in-pl...
5,make_key_frame,"(train: 'pd.DataFrame', test: 'pd.DataFrame')",13列すべてを「厳密値のまま」整数キー化したフレームを返す (S6E8 のブレークスルー).
6,add_smooth_keys,"(keys: 'pd.DataFrame', df: 'pd.DataFrame', sca...",
7,smooth_key_names,"(scales=(10, 100, 1000, 10000), commute: 'bool...",
8,add_digit_features,"(df: 'pd.DataFrame', cols=None, ks=None) -> 'p...",
9,drop_constant_cols,"(frames, cols)",全フレームで定数の列を落とす (学習を遅くするだけなので)。残った列名を返す.


## モデル別 FE ファイルの関数一覧

In [3]:
rows = []
for short in ["lgbm", "xgb", "catboost", "realmlp"]:
    mod_name = f"03_fe_{short}"
    mod = load_fe(short)
    for n, f in vars(mod).items():
        if callable(f) and not n.startswith("_") and getattr(f, "__module__", "") == mod_name:
            rows.append((mod_name + ".py", n, (f.__doc__ or "").strip().split("\n")[0][:60]))
pd.DataFrame(rows, columns=["ファイル", "関数", "説明"])

,ファイル,関数,説明
0,03_fe_lgbm.py,load_data,Same read_csv flow as 02_bl_lgbm.py.
1,03_fe_lgbm.py,make_categorical,Give train/test a shared category set (LightGB...
2,03_fe_lgbm.py,make_key_frame,"Integer-coded copies of all 13 raw columns, us..."
3,03_fe_lgbm.py,add_arithmetic_meaningful,Domain-meaningful ratio / diff / sum features.
4,03_fe_lgbm.py,add_arithmetic_all_pairs,diff / ratio / sum over every numeric pair.
5,03_fe_lgbm.py,add_group_means,avg-style aggregates over standardised numeric...
6,03_fe_lgbm.py,add_digit_features,"`(x // 10**k) % 10` as an int8 column, for eve..."
7,03_fe_lgbm.py,add_smooth_keys,Coarser resolutions of the two high-cardinalit...
8,03_fe_lgbm.py,count_encode,Count / frequency encoding fitted on train+tes...
9,03_fe_lgbm.py,target_encode_fold,Leak-free target encoding.


## 主要な関数の中身を読む

`03_fe_all.py` をここにコピーはしない。コピーすると `.py` を直したときノートブックが
置き去りになるため(実際この NB の `import fe_all` は `src/` へ移動した時点から壊れていた)。
代わりに `inspect.getsource()` で**実ファイルからそのまま表示**する。
`.py` を直せば次の実行で自動的に追従する。

In [4]:
import inspect


def show_source(func_name, mod=None):
    """src/03_fe_all.py の関数を実ファイルから読んで表示する。"""
    mod = mod or fe_all
    print(f"# {mod.__name__}.py :: {func_name}")
    print("-" * 78)
    print(inspect.getsource(getattr(mod, func_name)))


# ここを書き換えれば他の関数も読める(関数名は上の一覧表を参照)
show_source("target_encode_fold")

# 03_fe_all.py :: target_encode_fold
------------------------------------------------------------------------------
def target_encode_fold(keys_fit: pd.DataFrame, y_fit, other_frames, cols,
                       smooths=(20.0,), n_inner: int = 5, seed: int = 42):
    """リークフリー TE。戻り値 (te_fit, [te_other, ...])。

    keys_fit     : 現在の outer fold の**学習行**のキーフレーム
    other_frames : 同じ統計を当てるフレーム (通常 [valid_keys, test_keys])
    smooths      : float または "auto" のリスト。複数指定で Triple TE。
    """
    smooths = list(smooths) if isinstance(smooths, (list, tuple)) else [smooths]
    multi = len(smooths) > 1
    y_fit = np.asarray(y_fit)
    prior = float(y_fit.mean())
    te_fit = pd.DataFrame(index=keys_fit.index)
    te_others = [pd.DataFrame(index=f.index) for f in other_frames]

    inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=seed)
    inner_splits = list(inner.split(np.zeros(len(y_fit)), y_fit))

    for c in cols:
        arr = keys_fit[c].to_numpy()
        names = {s

**TE の要点はこの 3 つ。**

1. `keys_fit` は **fold の学習データだけ**。検証・テストの行は統計に一切入らない。
2. 学習行には**内側 CV の out-of-fold 値**を当てる(`n_inner`)。
   自分のラベルを含まない値になるので、学習時だけ TE が当たりすぎる楽観バイアスが消える。
   これは保険ではなく、XGBoost で **+0.00108** の実利があった。
3. `smooths` にリストを渡すと **1 キーから複数列**が生える(Triple TE)。
   縮約の強さ違いを同時に見せると、木がキーごとに適切な強さを選べる。

In [5]:
show_source("count_encode")

# 03_fe_all.py :: count_encode
------------------------------------------------------------------------------
def count_encode(keys_tr: pd.DataFrame, keys_te: pd.DataFrame, cols, freq: bool = True):
    out_tr = pd.DataFrame(index=keys_tr.index)
    out_te = pd.DataFrame(index=keys_te.index)
    n_total = len(keys_tr) + len(keys_te)
    for c in cols:
        vc = pd.concat([keys_tr[c], keys_te[c]], ignore_index=True).value_counts()
        a = keys_tr[c].map(vc).astype("float32").to_numpy()
        b = keys_te[c].map(vc).astype("float32").to_numpy()
        if freq:
            a, b = a / n_total, b / n_total
        out_tr[f"cnt_{c}"] = a
        out_te[f"cnt_{c}"] = b
    return out_tr, out_te



Count Encoding は**目的変数を使わない**ので `train` + `test` をまとめて数えてよい。
リークしないうえ、テストにしか出ない値の頻度も正しく入る。TE とは情報源が別なので加算的に効く
(ただし CatBoost は内部の Ordered Target Statistics と重複するため無効)。

In [6]:
show_source("catify")

# 03_fe_all.py :: catify
------------------------------------------------------------------------------
def catify(tr: pd.DataFrame, te: pd.DataFrame, cols=None, mode: str = "category"):
    """低カーデ数値列をカテゴリ扱いに変換した (tr, te) を返す."""
    cols = LOWCARD_NUM_COLS if cols is None else cols
    tr, te = tr.copy(), te.copy()
    for c in cols:
        if mode == "str":
            tr[c] = tr[c].astype(str)
            te[c] = te[c].astype(str)
        else:
            # XGBoost は float dtype のカテゴリを受け付けないので、必ず整数コードに
            # 変換してから category dtype にする (LightGBM もこれで問題ない)。
            cats = pd.Index(sorted(set(tr[c].unique()) | set(te[c].unique())))
            codes_tr = pd.Categorical(tr[c], categories=cats).codes.astype("int16")
            codes_te = pd.Categorical(te[c], categories=cats).codes.astype("int16")
            allc = pd.Index(range(len(cats)))
            tr[c] = pd.Categorical(codes_tr, categories=allc)
            te[c] = pd.Categorical(codes_te, categories=allc)
    retu

`catify` は **CatBoost 専用で +0.00170** と単体最大だった施策。
`Age`(45値)や `Charging_Stations_Near_Home`(15値)のような低カーディナリティの数値列を
「大小関係のある数」ではなく「ラベル」として扱わせる。CatBoost は Ordered Target Statistics で
カテゴリを処理するため、順序を捨てた方が情報が増える。
**同じことを LightGBM でやると -0.00016 で不採用**。モデルによって効く FE が違う典型例。

## 動作確認 — 小さなサンプルで出力を見る

実際にどんな列が作られるかを確認する。

In [7]:
train = pd.read_csv("data/train.csv").head(2000)
test = pd.read_csv("data/test.csv").head(500)
train.head(3)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes


### digit features — 数値を桁ごとの列にする

年収 84,880 なら 1の位=0、10の位=8、100の位=8… と分解する。
既定のビン数では潰れてしまう細かい値の違いを木に見せる狙い。

**検証結果**: LightGBM / XGBoost では効果なし(+0.0001)、**CatBoost のみ +0.00061**。
CatBoost は既定のビン数が 64 と粗く、年収の値の違いが最も潰れていたため。

In [8]:
fe_lgbm = load_fe("lgbm")
d = fe_lgbm.add_digit_features(train)
print("生成された列数:", d.shape[1])
print("列名:", list(d.columns))
d.head()

生成された列数: 15
列名: ['Age_digit0', 'Age_digit1', 'Annual_Income_USD_digit0', 'Annual_Income_USD_digit1', 'Annual_Income_USD_digit2', 'Annual_Income_USD_digit3', 'Daily_Commute_km_digit-1', 'Daily_Commute_km_digit0', 'Daily_Commute_km_digit1', 'Number_of_Cars_Owned_digit0', 'Charging_Stations_Near_Home_digit0', 'Charging_Stations_Near_Home_digit1', 'Charging_Stations_Near_Work_digit0', 'Charging_Stations_Near_Work_digit1', 'Environmental_Concern_Level_digit0']


,Age_digit0,Age_digit1,Annual_Income_USD_digit0,Annual_Income_USD_digit1,Annual_Income_USD_digit2,Annual_Income_USD_digit3,Daily_Commute_km_digit-1,Daily_Commute_km_digit0,Daily_Commute_km_digit1,Number_of_Cars_Owned_digit0,Charging_Stations_Near_Home_digit0,Charging_Stations_Near_Home_digit1,Charging_Stations_Near_Work_digit0,Charging_Stations_Near_Work_digit1,Environmental_Concern_Level_digit0
0,6,6,7,8,8,2,4,3,2,2,3,0,7,0,1
1,8,3,0,0,0,0,0,5,0,1,2,0,2,0,4
2,6,2,9,8,3,4,8,6,3,1,8,0,5,1,5
3,6,6,0,8,5,3,7,3,2,2,6,0,9,0,3
4,4,5,8,9,8,7,8,0,5,1,2,0,3,0,3


## 各モデルが実際に使っている列(本番構成)

ここまでは「どんな関数があるか」のカタログだった。ここからは **その関数を組み合わせた結果、
各モデルが最終的に何列を学習に渡しているか**を見る。

列名は `src/04_fe_run_<model>.py` の `--dump-features` で吐き出したものを読み込む。
このフラグは **fold 1 の学習行列を組み上げた直後に列名を JSON へ書いて終了する**(学習しない)ので、
本番とまったく同じコードパスを通った列一覧になる。再生成は数十秒:

```bash
uv run src/04_fe_run_lgbm.py --patterns base,te1,cnt1,digit,sk --smooths auto,10,100   --sample 0.02 --folds 1 --dump-features --tag lgbm
uv run src/04_fe_run_xgb.py --pattern tte_sk_dig --sample 0.02 --folds 1 --dump-features --out-suffix ""
uv run src/04_fe_run_catboost.py --fe te_all,catify,digits,skeys,te3 --folds 5 --rows 15000   --fast --dump-features --tag catboost
uv run src/04_fe_run_realmlp.py --folds 1 --subsample 0.02 --dump-features --tag realmlp
```

In [9]:
import feature_dump as fd

TAGS = ["lgbm", "xgb", "catboost", "realmlp"]
dumps = {t: fd.load(t) for t in TAGS}

for t in TAGS:
    d = dumps[t]
    print(f"{d['model']:<9} {d['n_features']:>3} 列   "
          f"(うち生の列 13 / 新規 {d['n_features'] - 13})   {d['note']}")

LightGBM   92 列   (うち生の列 13 / 新規 79)   patterns=base,te1,cnt1,digit,sk smooths=auto,10,100
XGBoost    93 列   (うち生の列 13 / 新規 80)   pattern=tte_sk_dig
CatBoost   80 列   (うち生の列 13 / 新規 67)   fe=te_all,catify,digits,skeys,te3
RealMLP    38 列   (うち生の列 13 / 新規 25)   exact_te=False digits=False no_orig=False


### 種類別の内訳 — どのモデルが何を使っているか

分類規則は `src/feature_dump.py` の `classify()`(列名の接頭辞・接尾辞で判定)。
ノートブックと実行スクリプトで同じ定義を共有している。

In [10]:
import pandas as pd

cross = pd.DataFrame({
    dumps[t]["model"]: pd.Series([fd.classify(c) for c in dumps[t]["columns"]]).value_counts()
    for t in TAGS
}).fillna(0).astype(int)
cross.loc["合計"] = cross.sum()
cross

,LightGBM,XGBoost,CatBoost,RealMLP
Count / Frequency,13,13,0,1
Smooth Keys(粗い解像度),0,0,0,4
Target Encoding,51,51,51,2
catify(数値→カテゴリ),0,0,0,7
digit / 小数の分解,15,16,16,1
① 生の列,13,13,13,13
ビン分割,0,0,0,5
フラグ,0,0,0,1
交互作用キー,0,0,0,2
元データ由来,0,0,0,1


読み取れること:

- **GBDT 3種は Target Encoding が列数の半分以上**(51列)。TE が最大の改善要因だったことと一致する。
- **Count Encoding は CatBoost にだけ無い**。CatBoost は内部の Ordered Target Statistics が
  同じ情報を持っているため重複し、実測でも効果がなかった。
- **CatBoost の `digit` は 16 列で LightGBM より 1 多い**。小数第1位を取り出した
  `Daily_Commute_km_d-1` が LightGBM 側では定数扱いで落ちているため。
- **RealMLP だけまったく別の構成**。TE は 2 列しかなく、代わりに catify・ビン分割・
  Smooth Keys で「値を離散化して embedding に食わせる」設計になっている。
  GBDT と相関が低くアンサンブルで効くのは、この構成の違いによるもの。

### モデル別の全列一覧

新規作成分が何なのかを実際に並べて確認する。長いので種類ごとにまとめて表示する。

In [11]:
from collections import defaultdict

def show(tag, per_line=3):
    d = dumps[tag]
    groups = defaultdict(list)
    for c in d["columns"]:
        groups[fd.classify(c)].append(c)
    print("=" * 78)
    print(f"{d['model']}  —  {d['n_features']} 列   [{d['note']}]")
    print("=" * 78)
    for g, cols in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        print(f"\n■ {g}  ({len(cols)} 列)")
        for i in range(0, len(cols), per_line):
            print("    " + "  ".join(f"{c:<34}" for c in cols[i:i + per_line]).rstrip())
    if d["cat_features"]:
        print(f"\n■ カテゴリとして渡している列 ({len(d['cat_features'])} 列)")
        for i in range(0, len(d["cat_features"]), per_line):
            print("    " + "  ".join(f"{c:<34}" for c in d["cat_features"][i:i + per_line]).rstrip())
    print()

show("lgbm")

LightGBM  —  92 列   [patterns=base,te1,cnt1,digit,sk smooths=auto,10,100]

■ Target Encoding  (51 列)
    te_Age_sauto                        te_Age_s10                          te_Age_s100
    te_Annual_Income_USD_sauto          te_Annual_Income_USD_s10            te_Annual_Income_USD_s100
    te_Daily_Commute_km_sauto           te_Daily_Commute_km_s10             te_Daily_Commute_km_s100
    te_Number_of_Cars_Owned_sauto       te_Number_of_Cars_Owned_s10         te_Number_of_Cars_Owned_s100
    te_Charging_Stations_Near_Home_sauto  te_Charging_Stations_Near_Home_s10  te_Charging_Stations_Near_Home_s100
    te_Charging_Stations_Near_Work_sauto  te_Charging_Stations_Near_Work_s10  te_Charging_Stations_Near_Work_s100
    te_Environmental_Concern_Level_sauto  te_Environmental_Concern_Level_s10  te_Environmental_Concern_Level_s100
    te_Gender_sauto                     te_Gender_s10                       te_Gender_s100
    te_City_Type_sauto                  te_City_Type_s10              

In [12]:
show("xgb")

XGBoost  —  93 列   [pattern=tte_sk_dig]

■ Target Encoding  (51 列)
    inc_f100_tea                        inc_f100_te10                       inc_f100_te100
    inc_f1000_tea                       inc_f1000_te10                      inc_f1000_te100
    inc_f10000_tea                      inc_f10000_te10                     inc_f10000_te100
    commute_f1_tea                      commute_f1_te10                     commute_f1_te100
    Age_tea                             Age_te10                            Age_te100
    Annual_Income_USD_tea               Annual_Income_USD_te10              Annual_Income_USD_te100
    Daily_Commute_km_tea                Daily_Commute_km_te10               Daily_Commute_km_te100
    Number_of_Cars_Owned_tea            Number_of_Cars_Owned_te10           Number_of_Cars_Owned_te100
    Charging_Stations_Near_Home_tea     Charging_Stations_Near_Home_te10    Charging_Stations_Near_Home_te100
    Charging_Stations_Near_Work_tea     Charging_Stations_Near_Wor

In [13]:
show("catboost")

CatBoost  —  80 列   [fe=te_all,catify,digits,skeys,te3]

■ Target Encoding  (51 列)
    te10_Age                            te20_Age                            te100_Age
    te10_Annual_Income_USD              te20_Annual_Income_USD              te100_Annual_Income_USD
    te10_Daily_Commute_km               te20_Daily_Commute_km               te100_Daily_Commute_km
    te10_Number_of_Cars_Owned           te20_Number_of_Cars_Owned           te100_Number_of_Cars_Owned
    te10_Charging_Stations_Near_Home    te20_Charging_Stations_Near_Home    te100_Charging_Stations_Near_Home
    te10_Charging_Stations_Near_Work    te20_Charging_Stations_Near_Work    te100_Charging_Stations_Near_Work
    te10_Environmental_Concern_Level    te20_Environmental_Concern_Level    te100_Environmental_Concern_Level
    te10_Gender                         te20_Gender                         te100_Gender
    te10_City_Type                      te20_City_Type                      te100_City_Type
    te10_Current_C

CatBoost の「① 生の列」13 列のうち 7 列は、上の **カテゴリとして渡している列** に
含まれている。これが **catify**(低カーディナリティの数値列をカテゴリ扱いにする)で、
列名は変わらないため列数には現れないが CatBoost 単体で +0.00170 と最大の効果があった施策。

In [14]:
show("realmlp")

RealMLP  —  38 列   [exact_te=False digits=False no_orig=False]

■ ① 生の列  (13 列)
    Age                                 Annual_Income_USD                   Daily_Commute_km
    Number_of_Cars_Owned                Charging_Stations_Near_Home         Charging_Stations_Near_Work
    Environmental_Concern_Level         Gender                              City_Type
    Current_Car_Type                    Home_Charging_Possible              Subsidy_Available
    Range_Anxiety_Level

■ catify(数値→カテゴリ)  (7 列)
    Age_cat_                            Annual_Income_USD_cat_              Daily_Commute_km_cat_
    Number_of_Cars_Owned_cat_           Charging_Stations_Near_Home_cat_    Charging_Stations_Near_Work_cat_
    Environmental_Concern_Level_cat_

■ ビン分割  (5 列)
    Annual_Income_USD_400_quantile_bin_  Annual_Income_USD_600_quantile_bin_  Annual_Income_USD_800_quantile_bin_
    Annual_Income_USD_900_quantile_bin_  Annual_Income_USD_1100_quantile_bin_

■ Smooth Keys(粗い解像度)  (4 列)
    Income_/_

### Target Encoding の 51 列は何のキーから作られているか

TE は「キー × smooth の強さ」で列が増える。smooth を 3 通り(`auto` / `10` / `100`)同時に
入れる **Triple TE** を使っているので、**キー 17 本 × 3 = 51 列**になっている。

キー 17 本の内訳は「生の 13 列」+「Smooth Key 4 本」。Smooth Key は年収・通勤距離を
粗く丸めた値で、厳密値だと 1 値あたりの行数が足りずに縮約が効きすぎる問題を補う。

なお **列名の付け方はモデルごとにバラバラ**(`te_Age_sauto` / `Age_tea` / `te10_Age`)なので、
`feature_dump.te_key_of()` で正規化してから突き合わせている。

In [15]:
te_keys = {}
for t in ["lgbm", "xgb", "catboost"]:
    ks = [fd.te_key_of(c) for c in dumps[t]["columns"] if fd.classify(c) == "Target Encoding"]
    te_keys[dumps[t]["model"]] = sorted(set(ks))
    print(f'{dumps[t]["model"]:<9} TE 列 {len(ks):>2} = キー {len(set(ks))} 本 x smooth 3 通り')

rows = []
for i in range(17):
    rows.append([te_keys[m][i] if i < len(te_keys[m]) else "" for m in te_keys])
te_tbl = pd.DataFrame(rows, columns=list(te_keys))
te_tbl.index = [f"key {i + 1}" for i in range(len(te_tbl))]
te_tbl

LightGBM  TE 列 51 = キー 17 本 x smooth 3 通り
XGBoost   TE 列 51 = キー 17 本 x smooth 3 通り
CatBoost  TE 列 51 = キー 17 本 x smooth 3 通り


,LightGBM,XGBoost,CatBoost
key 1,Age,Age,Age
key 2,Annual_Income_USD,Annual_Income_USD,Annual_Income_USD
key 3,Charging_Stations_Near_Home,Charging_Stations_Near_Home,Charging_Stations_Near_Home
key 4,Charging_Stations_Near_Work,Charging_Stations_Near_Work,Charging_Stations_Near_Work
key 5,City_Type,City_Type,City_Type
key 6,Current_Car_Type,Current_Car_Type,Current_Car_Type
key 7,Daily_Commute_km,Daily_Commute_km,Daily_Commute_km
key 8,Environmental_Concern_Level,Environmental_Concern_Level,Environmental_Concern_Level
key 9,Gender,Gender,Gender
key 10,Home_Charging_Possible,Home_Charging_Possible,Home_Charging_Possible


上 13 行は 3 モデルとも同じ生の列。**違うのは下 4 行の Smooth Key の刻み方**で、
LightGBM は年収 /10・/100・/1000、XGBoost は /100・/1000・/10000、CatBoost は /1・/100・/1000 と
各リーダーが別々にチューニングした結果ずれている。これも予測の非相関性に効いている。

### モデル間の共通列・固有列

アンサンブルの多様性がどこから来ているかの確認。

注意: **同じ FE でもモデルによって列名が違う**(LightGBM の `Age_digit0` と XGBoost の `Age_d0` は
同じもの)。そのため「固有」の列数は実際の中身の違いより大きく出る。
中身の違いは 1 つ上の **種類別の内訳** の表で見るのが正しい。

In [16]:
sets = {dumps[t]["model"]: set(dumps[t]["columns"]) for t in TAGS}
common = set.intersection(*sets.values())
print(f"4モデル全部に共通する列: {len(common)} 列  (= 生の列 13 のみ)\n")

for name, s in sets.items():
    others = set.union(*[v for k, v in sets.items() if k != name])
    only = sorted(s - others)
    print(f"{name:<9} 固有 {len(only):>2} 列: {', '.join(only[:6])}{' …' if len(only) > 6 else ''}")

4モデル全部に共通する列: 13 列  (= 生の列 13 のみ)

LightGBM  固有 79 列: Age_digit0, Age_digit1, Annual_Income_USD_digit0, Annual_Income_USD_digit1, Annual_Income_USD_digit2, Annual_Income_USD_digit3 …
XGBoost   固有 64 列: Age_ce, Age_te10, Age_te100, Age_tea, Annual_Income_USD_ce, Annual_Income_USD_te10 …
CatBoost  固有 51 列: te100_Age, te100_Annual_Income_USD, te100_Charging_Stations_Near_Home, te100_Charging_Stations_Near_Work, te100_City_Type, te100_Current_Car_Type …
RealMLP   固有 25 列: Age_Range_Anxiety_Level_, Age_cat_, Annual_Income_USD_1100_quantile_bin_, Annual_Income_USD_400_quantile_bin_, Annual_Income_USD_600_quantile_bin_, Annual_Income_USD_800_quantile_bin_ …


## 効果が確認できた関数・できなかった関数

詳細は `Log.md` の「Feature Engineering 検証結果」表を参照(なぜ試したか / 期待した効果 /
考察まで記録してある)。

**効いたもの**
- 厳密値 Target Encoding(入れ子 CV 版) — 最大の改善要因(+0.003 前後)
- Count / Frequency Encoding — LightGBM・XGBoost で有効、CatBoost では無効
- Triple TE + Smooth Keys — +0.0005〜0.001
- catify(低カーデ数値をカテゴリ扱い) — **CatBoost 固有**(+0.0017)。他モデルでは逆効果

**効かなかったもの**
- 四則演算(diff / ratio / sum / avg) — 3モデルすべてで無効〜悪化
- 交互作用 TE(2列 → 3列 → 6列 → 13列) — すべて無効
- 行フィンガープリント — train 全 668,665 行がユニークなため原理的に機能しない